# Anonymizer Pipeline — Debug Notebook

Executes each pipeline step individually so output can be inspected between steps.

**Steps**
1. Setup
2. Load text
3. `normalize()` — static term replacements
4. `ner()` — named entity recognition
5. `replace_groups()` — group placeholders
6. `correct()` — grammar correction
7. Full pipeline comparison

## 1 — Setup

In [ ]:
import sys, os, json, difflib
sys.path.insert(0, "../src")

from news_analyser.anonymizer import SpacyStrategy  # .env wird automatisch geladen

strategy = SpacyStrategy()
print("Strategy ready:", type(strategy).__name__)

## 2 — Load text

Loads from `data/debug_last_run/00_original_text.txt` by default.  
Replace `TEXT_SOURCE` with a custom string or another file path if needed.

In [ ]:
TEXT_SOURCE = "../data/debug_last_run/00_original_text.txt"

if os.path.isfile(TEXT_SOURCE):
    with open(TEXT_SOURCE, encoding="utf-8") as f:
        original_text = f.read()
else:
    original_text = TEXT_SOURCE  # treat as literal text

word_count = len(original_text.split())
print(f"{len(original_text)} Zeichen | {word_count} Wörter")
print("---")
print(original_text[:500], "..." if len(original_text) > 500 else "")

## 3 — Normalize

Replaces ideologically loaded terms with neutral equivalents (static lookup, no placeholders).

In [ ]:
normalized_text, norm_mapping = strategy.normalize(original_text)

print(f"Ersetzungen: {len(norm_mapping)}")
for replacement, original in norm_mapping.items():
    print(f"  '{original}' → '{replacement}'")

print()
diff = list(difflib.unified_diff(
    original_text.splitlines(),
    normalized_text.splitlines(),
    lineterm="", n=1
))
if diff:
    print("\n".join(diff))
else:
    print("(keine Änderungen)")

## 4 — NER

Detects named entities (PER, ORG, LOC) and replaces them with typed placeholders.

In [ ]:
ner_text, ner_mapping = strategy.ner(normalized_text)

print(f"Entitäten erkannt: {len(ner_mapping)}")
for placeholder, surface in ner_mapping.items():
    print(f"  {placeholder} → '{surface}'")

print()
diff = list(difflib.unified_diff(
    normalized_text.splitlines(),
    ner_text.splitlines(),
    lineterm="", n=1
))
if diff:
    print("\n".join(diff))
else:
    print("(keine Änderungen)")

## 5 — Replace groups

Replaces group identifiers with `Gruppe-X` placeholders.  
Set `GROUP_TERMS` to a list from Pass 0 or leave empty.

In [ ]:
# Load group terms from Pass 0 output if available, otherwise leave empty.
_terms_file = "../data/debug_last_run/01_detected_terms.json"
if os.path.isfile(_terms_file):
    with open(_terms_file, encoding="utf-8") as f:
        GROUP_TERMS = json.load(f)
    print(f"Geladene group_terms: {GROUP_TERMS}")
else:
    GROUP_TERMS = []
    print("Keine group_terms (01_detected_terms.json nicht gefunden)")

groups_text, group_mapping = strategy.replace_groups(ner_text, GROUP_TERMS)

print(f"\nGruppen ersetzt: {len(group_mapping)}")
for placeholder, term in group_mapping.items():
    print(f"  {placeholder} → '{term}'")

print()
diff = list(difflib.unified_diff(
    ner_text.splitlines(),
    groups_text.splitlines(),
    lineterm="", n=1
))
if diff:
    print("\n".join(diff))
else:
    print("(keine Änderungen)")

## 6 — Correct

Fixes grammatical errors introduced by earlier replacements (article agreement, pronouns).  
Currently a no-op in `SpacyStrategy` — placeholder for future grammar corrections.

In [ ]:
corrected_text = strategy.correct(groups_text)

diff = list(difflib.unified_diff(
    groups_text.splitlines(),
    corrected_text.splitlines(),
    lineterm="", n=1
))
if diff:
    print("\n".join(diff))
else:
    print("(keine Änderungen — correct() ist noch no-op)")

## 7 — Full pipeline comparison

Runs `anonymize()` in one call and compares the result against the step-by-step output.

In [ ]:
result = strategy.anonymize(original_text, GROUP_TERMS)

match = result["text"] == corrected_text
print(f"Pipeline-Ergebnis == Schrittweises Ergebnis: {match}")

print(f"\nGesamtes Mapping ({len(result['mapping'])} Einträge):")
for k, v in result["mapping"].items():
    print(f"  {k} → '{v}'")

print("\n--- Finaler Text ---")
print(result["text"])